In [5]:
import torch, torch.nn as nn, json
from torchvision import models
from helper_functions.data_loading.data_utils import get_dataloaders
from models.baseline import BaselineCNN
DATA_DIR = "dataset/garbage_classification"

In [6]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
train_loader, val_loader, test_loader, class_names, class_weights = get_dataloaders(DATA_DIR)

In [7]:
# baseline
baseline = BaselineCNN(num_classes=len(class_names))
baseline.load_state_dict(torch.load("models/baseline_cnn.pt", map_location=device))
baseline = baseline.to(device).eval()

# mobilenet
mobilenet = models.mobilenet_v2()
mobilenet.classifier[1] = nn.Linear(mobilenet.last_channel, len(class_names))
mobilenet.load_state_dict(torch.load("models/mobilenet_v2.pt", map_location=device))
mobilenet = mobilenet.to(device).eval()

# resnet
resnet = models.resnet50()
resnet.fc = nn.Linear(resnet.fc.in_features, len(class_names))
resnet.load_state_dict(torch.load("models/resnet50.pt", map_location=device))
resnet = resnet.to(device).eval()

In [8]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

In [9]:
results = {}
for name, m in [("Baseline CNN", baseline), ("MobileNetV2", mobilenet), ("ResNet50", resnet)]:
    y_true, y_pred = get_predictions(m, test_loader)
    results[name] = {
        "y_true": y_true, "y_pred": y_pred,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }
    print(f"{name:14s} | acc {results[name]['accuracy']:.4f} "
          f"| macro-F1 {results[name]['macro_f1']:.4f} "
          f"| weighted-F1 {results[name]['weighted_f1']:.4f}")

Baseline CNN   | acc 0.7805 | macro-F1 0.7354 | weighted-F1 0.7801
MobileNetV2    | acc 0.9596 | macro-F1 0.9453 | weighted-F1 0.9595
ResNet50       | acc 0.9738 | macro-F1 0.9683 | weighted-F1 0.9738
